In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

data = [
    (1, 101, "250.50", "COMPLETED", "2025-01-10"),
    (2, 102, None, "PENDING", "2025-01-11"),
    (3, None, "300.00", "COMPLETED", "2025-01-12"),
    (4, 103, "450.00", None, "2025-01-13"),
    (5, 102, "200.00", "COMPLETED", None),
    (6, 104, None, "PENDING", "2025-01-15"),
    (7, 105, "500.00", "CANCELLED", "2025-01-16"),
    (7, 105, "500.00", "CANCELLED", "2025-01-16")
]

columns = ["order_id", "customer_id", "amount", "status", "order_date"]

bronze_orders = spark.createDataFrame(data, columns)

silver_orders = (
    bronze_orders
    .dropna(
        subset=["customer_id", "order_id"],
        how="any"
    )

    .dropDuplicates(["order_id"])
    
    .withColumn(
        "amount",
        F.col("amount").cast(DoubleType())
    )
    .withColumn(
        "status",
        F.lower(F.col("status"))
    )
    .withColumn(
        "order_date",
        F.col("order_date").cast(DateType())
    )
    .withColumn(
        "order_category",
        F.when(F.col("amount") >= 400, "High")
        .when(F.col("amount") >= 200, "Medium")
        .otherwise("Low")
    )
    .withColumn(
        "order_year",
        F.year(F.col("order_date"))
    )
    .withColumn(
        "order_month",
        F.month(F.col("order_date"))
    )

    .fillna({
        "amount": 0,
        "status": "unknown"
    })
)

display(silver_orders)